In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler , RobustScaler

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

df=sns.load_dataset('titanic')
print(df.shape)
df.head()

(891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [5]:
#data cleaning

print("Shape:", df.shape)
print("\ncolumns:", list(df.columns))
print("\ndtypes:\n", df.dtypes)

df["pclass"] = df["pclass"].astype("category")
df["survived"] = df["survived"].astype("bool")
df.dtypes[["pclass","survived"]]

Shape: (891, 15)

columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

dtypes:
 survived          int64
pclass            int64
sex                 str
age             float64
sibsp             int64
parch             int64
fare            float64
embarked            str
class          category
who                 str
adult_male         bool
deck           category
embark_town         str
alive               str
alone              bool
dtype: object


pclass      category
survived        bool
dtype: object

In [6]:
missing=df.isnull().sum().sort_values(ascending=False)
missing_pct=(missing/len(df)*100).round(2)
missing_summary=pd.DataFrame({"missing_count": missing, "missing_pct":missing_pct})
missing_summary[missing_summary['missing_count']>0]

,missing_count,missing_pct
deck,688,77.22
age,177,19.87
embarked,2,0.22
embark_town,2,0.22


In [8]:
df_clean=df.copy()
df_clean = df_clean.drop(columns=["deck"])
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())

for col in["embarked", "embark_town"]:
    df_clean[col]=df_clean[col].fillna(df_clean[col].mode()[0])

print("remaining missing values:\n", df_clean.isnull().sum().sum())


remaining missing values:
 0


In [9]:
n_dupes=df_clean.duplicated().sum()
print(f"duplicat rows found:{n_dupes}")
df_clean=df_clean.drop_duplicates()
print(f"shape after dropping duplicates:{df_clean.shape}")

duplicat rows found:116
shape after dropping duplicates:(775, 14)


In [11]:
messy_sex=df_clean["sex"].str.upper().sample(frac=0.3, random_state=1)
demo=df_clean["sex"].copy()
demo.loc[messy_sex.index]=messy_sex
print("before standardizing:",demo.unique())

standardized = demo.str.strip().str.lower().replace({"m":"male","f":"female"})
print("after standardizing:",standardized.unique())

before standardizing: <StringArray>
['MALE', 'female', 'FEMALE', 'male']
Length: 4, dtype: str
after standardizing: <StringArray>
['male', 'female']
Length: 2, dtype: str


In [ ]:
from scipy import stats

col="fare"

Q1, Q3=df_clean[col].quantile([0.25,0.75])
IQR= Q3-Q1
lower,upper=Q1-1.5*IQR, Q3+1.5*IQR
iqr_outliers=df_clean[(df_clean[col]<lower)|(df_clean[col]>upper)]

z-score=np.abs(stats.zscore(df_clean[col]))
